# Compute residual norm coefficients

This notebook computes the residual norm coefficients as part of the variable weights.

In [1]:
import os
import yaml
import copy
import numpy as np
import xarray as xr

In [2]:
from scipy.stats import gmean

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

## FE

In [4]:
# get variable information from data_preprocessing/config
config_name = os.path.realpath('data_config_FE.yml')

with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [5]:
N_levels = 4

base_dir = '/glade/derecho/scratch/ksha/FastEddy/FE_04lev_new/'
ds_example = xr.open_zarr(base_dir+'experiment_00.zarr')
level = np.array(ds_example['zIndex'])

In [6]:
# # get variable names
# varnames = list(conf['residual'].keys())
# varnames = varnames[:-5] # remove save_loc and others

# varname_upper = ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot', 'WRF_Q_tot_05']
# varname_surf = list(set(varnames) - set(varname_upper))

In [7]:
varname_upper = ['u', 'v', 'w', 'theta', 'rho', 'qv', 'TKE_0'] 
varname_surf = []

# 'WRF_precip_025', 'WRF_radar_composite_025', 'WRF_OLR', 'WRF_TCC', 'WRF_GLW', 'WRF_SWDOWN'

In [8]:
MEAN_values = {}
STD_values = {}

for varname in varname_surf:
    save_name = conf['residual']['save_loc'] + '{}_mean_std_{}.npy'.format(
        conf['residual']['prefix'], varname)
    mean_std = np.load(save_name)
    MEAN_values[varname] = mean_std[0]
    STD_values[varname] = mean_std[1]

for varname in varname_upper:

    # -------------------------------------------- #
    # allocate all levels
    mean_std_all_levels = np.empty((2, N_levels))
    mean_std_all_levels[...] = np.nan
    
    for i_level in range(N_levels):
        save_name = conf['residual']['save_loc'] + '{}_level{}_mean_std_{}.npy'.format(
            conf['residual']['prefix'], i_level, varname)
        mean_std = np.load(save_name)
        mean_std_all_levels[:, i_level] = mean_std

    # -------------------------------------------- #
    # save
    MEAN_values[varname] = np.copy(mean_std_all_levels[0, :])
    STD_values[varname] = np.copy(mean_std_all_levels[1, :])

In [9]:
# separate upper air (list) and surf (float) std values
N_upper = len(varname_upper)
std_val_all = list(STD_values.values())
#std_val_surf = np.array(std_val_all[:-N_upper])
std_val_upper = std_val_all[-N_upper:]

# combine
std_concat = std_val_upper #np.concatenate([std_val_surf]+ std_val_upper)

# geometrical mean (not used)
std_g = gmean(np.sqrt(std_concat))

In [10]:
# ------------------------------------------------------- #
# create xr.DataArray for std

# use the same level coord as mean
ds_std_6h = xr.Dataset(coords={'zIndex': level})

for varname, data in STD_values.items():
    data = np.sqrt(data) / std_g
    if len(data.shape) == 1:
        data_array = xr.DataArray(
            data,
            dims=["zIndex",],
            coords={"zIndex": level},
            name=varname,
        )
        ds_std_6h[varname] = data_array
    else:
        data_array = xr.DataArray(
            data,
            name=varname,
        )
        ds_std_6h[varname] = data_array

In [11]:
ds_std_6h.to_netcdf('/glade/derecho/scratch/ksha/FastEddy/mean_std/FE_residual_04lev_new.nc')

In [12]:
# ds_new = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_residual_1980_2019_12lev.nc')
# ds_W = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_residual_1980_2019_12lev_W.nc')
# ds_old = xr.open_dataset('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/mean_std/C404_residual_1980_2019_15lev_20250629.nc')

# for varname in ds_W.keys():
#     print(f'=================== {varname} ===================')
#     try:
#         print(ds_W[varname].values)
#         print(ds_new[varname].values)
#         print(ds_old[varname].values)
#     except:
#         pass